<p style='font: 600 36px "Lato", "Open Sans", "Helvetica Neue", Helvetica, Arial, sans-serif'>
    <img align="right" width="300" height="150" style="border-radius: 12px;" src="https://www.kaggle.com/competitions/154281/images/header">
    RSNA Knee Abnormality Detection [EDA]
    <p style='font: 500 22px "Lato", "Open Sans", "Helvetica Neue", Helvetica, Arial, sans-serif'>
        Create a model that can detect knee abnormalities based on multimodal imaging data
    </p>
</p>

<p style='font: 300 18px/22px "Lato", "Open Sans", "Helvetica Neue", Helvetica, Arial, sans-serif'>
    In this notebook, we'll try to understand what data we have to work with. We'll examine the data provided in the <code style="background-color: #f4f4f4; color: #d63384; padding: 2px 6px; border-radius: 4px; font-family: monospace; font-size: 15px;">train.csv</code> and <code style="background-color: #f4f4f4; color: #d63384; padding: 2px 6px; border-radius: 4px; font-family: monospace; font-size: 15px;">train_series.csv</code> files, and we'll also analyze the <code style="background-color: #f4f4f4; color: #d63384; padding: 2px 6px; border-radius: 4px; font-family: monospace; font-size: 15px;">metadata</code> content of the <code style="background-color: #f4f4f4; color: #d63384; padding: 2px 6px; border-radius: 4px; font-family: monospace; font-size: 15px;">DICOM</code> files
</p>
        


In [ ]:
import numpy as np
import pandas as pd
from kaggle_secrets import UserSecretsClient
import os
import re
import glob
from transformers import pipeline
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import pydicom
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option('display.max_colwidth', 300)

In [ ]:
MAIN_PATH = '/kaggle/input/competitions/rsna-knee-abnormality-detection/'
TRAIN_PATH = '/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series/'

In [ ]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

# Analyzing the train.csv file

In [ ]:
train_df = pd.read_csv(MAIN_PATH + 'train.csv')

In [ ]:
train_df.info()

## Labels

Labels are available for only 58 studies out of 4,500! Nevertheless, let's take a look at the class distribution

In [ ]:
feature_means = train_df.dropna()[train_df.columns[3:]].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=feature_means.values, y=feature_means.index)
plt.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="50% series")

plt.title("What proportion of series have each feature?")
plt.xlabel("rate")
plt.ylabel("feature")
plt.xlim(0, 1)
plt.legend();

## Reports statistic

Each study has a description (report). Based on these descriptions, the dataset will need to be annotated. An additional complication is that the descriptions may be in different languages.

Let's check how many descriptions are repeated and how extensive they are.

In [ ]:
tst_uniq = train_df[['StudyInstanceUID', 'Report']].groupby('Report').size().reset_index(name='count')

In [ ]:
tst_uniq[tst_uniq['count']>1].sort_values(by='count', ascending=False).head()

There are not many repetitions, and they mainly concern studies of healthy organs

In [ ]:
# reports length statistic
train_df['Report'].str.len().describe()

On average, a description contains about 1000 characters

## Report languages

Let's determine what languages we will have to deal with

In [ ]:
import fasttext
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="facebook/fasttext-language-identification", filename="model.bin")

In [ ]:
model = fasttext.load_model(model_path)

In [ ]:
texts = train_df['Report'].str.replace('\n', ' ', regex=False).replace('\r', '', regex=False).to_list()
res = model.predict(texts)

In [ ]:
languages = [(re.findall(r"__label__(.*?)_", tuple_lang_prob[0][0])[0], 
              tuple_lang_prob[1][0],
              re.findall(r"__label__(.*?)$", tuple_lang_prob[0][0])[0], 
             ) 
             for tuple_lang_prob in zip(res[0], res[1])
            ]

In [ ]:
lang_mapping = {
    'bos': 'Bosnian',  'bul': 'Bulgarian',   'deu': 'German',
    'ell': 'Greek',    'eng': 'English',     'fra': 'French',
    'glg': 'Galician', 'hrv': 'Croatian',    'kor': 'Korean',
    'nld': 'Dutch',    'por': 'Portuguese',  'spa': 'Spanish',
    'tur': 'Turkish',  'yue': 'Cantonese'
}

In [ ]:
wordcloud = WordCloud(
    width=800, 
    height=400, 
    background_color="white"
).generate(' '.join([lang_mapping[id[0]] for id in languages]))

plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off");

Anything from Cantonese and minor is not. Below, we'll consider situations where the model is less than 50% confident in its answer. All of these rare languages ​​will be included in this sample

In [ ]:
# if someone perceives it better in percentages )
lang_df = pd.DataFrame({
    'StudyInstanceUID': train_df['StudyInstanceUID'],
    'report': texts, 
    'language': [res[2] for res in languages], 
    'prob': [res[1] for res in languages],
})

(lang_df['language'].value_counts(normalize=True) * 100).round(1).astype(str) + '%'

In [ ]:
# here the model doubts the result
lang_doubt_df = lang_df[lang_df['prob']<0.5]
lang_doubt_df['language'].value_counts()

In [ ]:
lang_doubt_df[lang_doubt_df['language']=='kor_Hang']

For example, this "Korean" turned out to be English. You can check the others. It seems the reasons for this uncertainty lie in the short phrases and abundance of specialized terminology

In any case, we can confirm that we are dealing with a dozen European languages

We'll leave English and Spanish as is. We'll change Cantonese, Korean, and Croatian to English, and Portuguese and Galician to Spanish. We'll only change all of the above for the sample below 50%, and we'll change Cantonese entirely, as one row with a 57% probability was missing from the sample, but it's also incorrect.

In [ ]:
lang_df.loc[lang_df['language'] == 'yue_Hant', 'language'] = 'eng_Latn'
lang_df.loc[(lang_df['prob'] < 0.5) & (lang_df['language'].isin(['kor_Hang', 'hrv_Latn'])), 'language'] = 'eng_Latn'
lang_df.loc[(lang_df['prob'] < 0.5) & (lang_df['language'].isin(['por_Latn', 'glg_Latn'])), 'language'] = 'spa_Latn'

In [ ]:
# let's save it for the future
lang_df.drop('prob', axis=1).to_parquet('language.parquet', index=False)

# Analyzing the train_series.csv file

The train_series table contains three additional attributes for each series: 'Fluid_Sensitive', 'Fat_Suppression', and 'Anatomical_Plane'. The last attribute is simple—it's the imaging projection, which can be one of three options: `Coronal`, `Sagittal`, or `Axial`. The other two are helpful in image preprocessing, as they indicate that fluid and fat are being displayed in a specific way.

Let's determine the basic minimum:
1. How many series are there per study?
2. Are all projections available for each study?
3. What combinations of the binary attributes 'Fluid_Sensitive' and 'Fat_Suppression' are available for each projection?

In [ ]:
train_series_df = pd.read_csv(MAIN_PATH + 'train_series.csv')

In [ ]:
# there arn't any nulls
train_series_df.info()

First, let's determine what options for the number of series are available for all studies

In [ ]:
# number of Series per Study
ax = train_series_df[['StudyInstanceUID', 'SeriesInstanceUID']].groupby('StudyInstanceUID').count()['SeriesInstanceUID'].value_counts().plot.bar()

for container in ax.containers:
    ax.bar_label(container, padding=3)

More than half of the studies contained five series. It's also worth noting that no study contained fewer than three series, suggesting that each study contained images in all three projections. However, this is only an assumption that needs to be verified

In [ ]:
train_series_df[['StudyInstanceUID', 'Anatomical_Plane']].groupby('StudyInstanceUID').nunique()['Anatomical_Plane'].value_counts()

We've confirmed that for all 4,407 studies, there's at least one series of each of the three projections. At least one, but it's clear there could be more. Let's explore the options!

In [ ]:
# how many variants of one projection can there be in a series?
numb_of_planes = train_series_df[['StudyInstanceUID', 'Anatomical_Plane']].value_counts().reset_index(name='counts')

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=numb_of_planes, x='Anatomical_Plane', hue='counts', palette="muted");

In the diagram, we see that the maximum number of series in a single projection in a study can be up to seven! It's unclear why... They were probably just there ) We also see that there's usually one axial projection, and two coronal and sagittal projections.

We also have two binary attributes: `Fluid_Sensitive` and `Fat_Suppression`, which could be the reason for multiple series in a single projection. Since these are two binary attributes, they can explain the maximum of four series in a single projection. But there can be up to seven! Well, at least we'll try to understand something.

First, let's look at the possible combinations of `Fluid_Sensitive` and `Fat_Suppression`

In [ ]:
# number of fluid/fat signals (both signals occur only simultaneously)
(train_series_df[['Fluid_Sensitive', 'Fat_Suppression']].value_counts(normalize=True) * 100).round(1).astype(str) + '%'

And here's a surprise! There are only two combinations, not four as we initially assumed. Either both parameters are True, or both are False. The fat suppression and fluid sensitivity series account for just over half.

Let's look at this distribution by projection:

In [ ]:
((train_series_df[['Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression']].value_counts() / 
 train_series_df[['Anatomical_Plane']].value_counts()) * 100).round(1).astype(str) + '%'

Here we see that the distribution is roughly 50/50 for the coronal and sagittal projections, while for the axial projection, the images are predominantly (80%) fat-suppressed and fluid-emphasized.

Next, we'll calculate how many combinations of additional attributes are available for each study

In [ ]:
numb_of_planes_and_ext = train_series_df[['StudyInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive']].groupby(['StudyInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive']).size().reset_index(name='counts')
numb_of_planes_and_ext.head()

Let's look at the series that have all the same additional features, i.e. the same projection and the same combination of fat suppression and fluid sensitivity

In [ ]:
print('the number of repetitions of series with the same parameters in studies:')
print(f' - doubled             {len(numb_of_planes_and_ext[numb_of_planes_and_ext['counts']==2]):>4}')
print(f' - tripled             {len(numb_of_planes_and_ext[numb_of_planes_and_ext['counts']==3]):>4}')
print(f' - for and more times  {len(numb_of_planes_and_ext[numb_of_planes_and_ext['counts']>3]):>4}')

Well, quite a few... Let's look at an example. Let's select studies with more than three repetitions and examine images from one of them

In [ ]:
numb_of_planes_and_ext[numb_of_planes_and_ext['counts']>3]

Let's take as an example the first study in the list with index 4966, where there are five images in axial projection without fat suppression and fluid accentuation

In [ ]:
for_print_df = train_series_df[train_series_df['StudyInstanceUID'] == numb_of_planes_and_ext['StudyInstanceUID'][4966]]
for_print_df

We see that the study contains six axial images, five of which are without fat suppression, and one with suppression. Let's look at the images.

In [ ]:
def feat_by_series_uid(df, series_uid, prefix=TRAIN_PATH):
    """
    Returns a dict of features for a series
    df - dataframe based on train(test)_series.csv
    series_uid - unique series identifier
    prefix - part of the path to files
    """
    
    feat_dict =  df[df['SeriesInstanceUID']==series_uid].to_dict(orient='records')[0]
    feat_dict['path'] = TRAIN_PATH + '/'.join([feat_dict['StudyInstanceUID'], feat_dict['SeriesInstanceUID']]) + '/*.dcm'
    return feat_dict

def print_series_images(data, clmns_numb = 5, figsize=(16,20)):
    """
    Prints images of a series in the order they were scanned
    data - dict from feat_by_series_uid()
    clmns_numb - number of columns
    figsize - tuple for plt.subplots
    """
    series_path = glob.glob(data['path'])
    fig, axs = plt.subplots(int(np.ceil(len(series_path) / clmns_numb)), 
                            clmns_numb,
                            figsize=figsize,
                            constrained_layout=True,
                            subplot_kw={'xticks':[], 'yticks':[]})
    
    for i, path in enumerate(series_path):
        
        ds = pydicom.dcmread(path)
        img = ds.pixel_array
        idx = ds['InstanceNumber'].value - 1 # for sort by number in series
        axs[idx // clmns_numb, idx % clmns_numb].imshow(img, cmap='bone')

    # let's get some dicom metadata
    manufacturer = getattr(ds, 'Manufacturer', "Unknown")
    magn_field_strng = getattr(ds, 'MagneticFieldStrength', "Unknown")
    tr = getattr(ds, 'RepetitionTime', "Unknown")
    te = getattr(ds, 'EchoTime', "Unknown")
    fig.suptitle('{0}\n{1}, Fluid_Sensitive - {2}, Fat_Suppression - {3}\nManufacturer: {4}, Field strength - {5}\nRepetition Time - {6}, Echo Time {7}'.format(
        data['SeriesInstanceUID'],   # 0
        data['Anatomical_Plane'],    # 1
        data['Fluid_Sensitive'],     # 2
        data['Fat_Suppression'],     # 3
        manufacturer,                # 4
        magn_field_strng,            # 5
        tr,                          # 6
        te,                          # 7
    )
                )

The first series in the axial projection without fat suppression and fluid sensitivity (index 5678)

In [ ]:
print_series_images(feat_by_series_uid(train_series_df, for_print_df['SeriesInstanceUID'][5678]))

The next series uses the same parameters. Note that, despite the similar parameters, the color gamut is significantly different. It's obvious that these images have different weighting. Unfortunately, it wasn't provided in the train_series file, so we'll have to decipher the dicom file metadata.

In [ ]:
print_series_images(feat_by_series_uid(train_series_df, for_print_df['SeriesInstanceUID'][5679]))

Well, here's another series of this study, with the same parameters. This one is more similar to the first series, but there are still some differences

In [ ]:
print_series_images(feat_by_series_uid(train_series_df, for_print_df['SeriesInstanceUID'][5680]))

Let's take another look at the same projection, but with fat suppression and fluid emphasis. It's clearly different from all three previous ones...

In [ ]:
print_series_images(feat_by_series_uid(train_series_df, for_print_df['SeriesInstanceUID'][5683]))

# DICOM metadata

Next, we'll move on to examining a broader set of metadata that we'll extract from DICOM files. First, let's determine what this metadata set is and whether it's the same for all images

In [ ]:
import polars as pl
import random
from tqdm.notebook import tqdm
from tqdm.contrib.concurrent import thread_map
from tqdm.asyncio import tqdm as async_tqdm

In [ ]:
%%time
file_list = glob.glob(TRAIN_PATH + '/**/*.dcm', recursive=True)
print(f'Total train images - {len(file_list):,d}')

In [ ]:
def attr_list(dcm_file):

    dcm_ds = pydicom.dcmread(dcm_file, stop_before_pixels=True)
    df = pl.DataFrame({'file':dcm_file, 'feat_list':[dcm_ds.dir()], 'hash_fl':hash(tuple(dcm_ds.dir()))})
    meta_data.vstack(df, in_place=True)

In [ ]:
rate = 0.05 # if you want to take only sample 

meta_data = pl.DataFrame(schema={
    'file':pl.String, 
    'feat_list':pl.List(pl.String),
    'hash_fl':pl.Int64
})

if rate < 1:
    k = int(len(file_list) * rate)
    file_sample = random.sample(file_list, k)
else:
    file_sample = file_list
    
_ = thread_map(attr_list, [dcm_file for dcm_file in file_sample],
               tqdm_class=async_tqdm, max_workers=32, total=len(file_sample), ncols=100)

In [ ]:
# let's get some stat of features sets
feat_sets_stat = (
    meta_data.group_by('feat_list', 'hash_fl').agg(pl.col('hash_fl').len().alias('count'))
    .sort('count', descending=True)
    .with_row_index(name="index")
    .with_columns(
        pl.col('feat_list').list.len().alias('feat_len'), 
        (pl.col("count").cum_sum() / pl.col("count").sum()).alias("cum_percentage")
    )
)

In [ ]:
feat_sets_stat

In [ ]:
print(f'Number of metadata list variants - {feat_sets_stat.height}')
print(f'Number of list length options - {feat_sets_stat['feat_len'].unique().len()}')

To speed things up, I took a random sample of 5% and got 83 metadata set variants. Someone didn't pay much attention to consistency... If you don't mind the time, you can set `rate=1`, and in 45 minutes you'll get all 86 variants. Not much more )

In [ ]:
feat_sets_stat['feat_len'].describe()

The sets of props also vary greatly in length: a minimum of 30, a maximum of 74. Therefore, these sets will intersect at a maximum of 30 elements. Let's check...

In [ ]:
def feat_set_intersection(df):
    common_feat = list(set.intersection(*map(set, df)))
    
    print(f"The number of common features for all {df.shape[0]} lists is: {len(common_feat)}")
    print("List of common features:", common_feat)
    
feat_set_intersection(feat_sets_stat['feat_list'])

Only these 28 features are guaranteed to be present for all series.

Perhaps it's also worth looking at the quantitative ratio of all the sets. For this, it's best to create a set for all files, but let's trust the statistics of a random selection.

In [ ]:
feat_sets_stat.select(['index', 'cum_percentage']).plot.line(x='index', y='cum_percentage')

The first thirty sets represent 95% of the total sample size. Let's check if they intersect in more elements

In [ ]:
feat_set_intersection(feat_sets_stat['feat_list'][:30])

No, the same 28 features. Let's try to group them by their usefulness for further pre-processing

1. **Geometry and resampling (may be required)**
- PixelSpacing: Pixel size. Needed for resampling along the X and Y axes.
- SliceThickness: Slice thickness. Needed for resampling along the Z axis when assembling the 3D volume.
- ImagePositionPatient: Coordinates of the upper left corner of the image. Allows for proper slice ordering.
- ImageOrientationPatient: Coordinate direction vectors. Needed for projection verification, but we've already defined one.
- SliceLocation: Relative slice position. Used as an alternative or supplement to slice sorting.
2. **Brightness and contrast normalization (not needed, as these nuances will be taken into account when reading using pixel_array)**
- WindowCenter and WindowWidth: Center and width of the visualization window
- BitsAllocated: Number of bits allocated per pixel
- BitsStored: Number of bits actually used
- HighBit: The most significant bit, which determines the sign or significant part
- PixelRepresentation: Determines whether the data is signed (1) or unsigned (0)
3. **Data format and dimensions (may be needed)**
- Rows and Columns: Height and width of the matrix in pixels (for resizing)
- SamplesPerPixel: Number of color channels (not needed, the dataset only has one channel)
- PhotometricInterpretation: Color model. Specifies how to interpret the channels (the train data only shows MONOCHROME2, but MONOCHROME1 should also be considered in case it appears in the test data).
4. **Data Filtering and Grouping**
- Modality: Study Type (train only shows MR)
- ImageType: Image Type. Allows you to filter out secondary (DERIVED) or service images, leaving only original (ORIGINAL) images.
- SeriesInstanceUID: Series ID (not needed, the structure is organized by series)
- InstanceNumber: Image number in the series. I used this above to sort images during visualization (ImagePositionPatient is said to be more reliable).
5. **Less useful preprocessing attributes:** PatientID, StudyInstanceUID, SOPInstanceUID, SOPClassUID, FrameOfReferenceUID, Manufacturer, ManufacturerModelName, AcquisitionNumber, and SeriesNumber. Manufacturer and ManufacturerModelName can be used for specific equipment, as we have images from different MRI machines. However, this isn't for preprocessing, but for troubleshooting.

But I'd like to see something else... We've already noted the different weightings above, which can be worked with using `Repetition Time`, `Echo Time`, and `ScanningSequence`. Let's see how fully they are represented in the dataset.

In [ ]:
def repr_feats(feat_list: list):
    
    feat_extr = [pl.col('feat_list').list.contains(name) for name in feat_list]
    intercept_slope_sets = feat_sets_stat.filter(pl.all_horizontal(feat_extr))

    print(f'Number of sets with this feature - {intercept_slope_sets.shape[0]}')
    print(f'Share of the total number of images in the sample - {(intercept_slope_sets['count'].sum() / feat_sets_stat['count'].sum()):.1%}')

In [ ]:
repr_feats(['RepetitionTime', 'EchoTime'])

In [ ]:
repr_feats(['ScanningSequence'])

Very good! These features are available for almost all series. Let's also grab the `SeriesDescription`; it often contains a lot of useful information, but it needs to be parsed

In [ ]:
repr_feats(['SeriesDescription'])

We're trying to collect metadata. We could try doing it on a sample, but I decided to be savvy and did it on all 820,000 images! )

In [ ]:
# let's stop at this list, we have to stop somewhere )
feat_list = [
    'Manufacturer',
    'ManufacturerModelName',
    'PixelSpacing', 
    'SliceThickness', 
    'ImagePositionPatient', 
    'Rows',
    'Columns',
    'PhotometricInterpretation',
    'InstanceNumber',
    # not all fields below are available for all images
    'RepetitionTime',
    'EchoTime',
    'ScanningSequence',
    'SeriesDescription',
]

In [ ]:
# take some of the data from train_series
ext_data = (train_series_df
            [['SeriesInstanceUID', 'Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane']]
            .set_index('SeriesInstanceUID').to_dict(orient='index')
           )

In [ ]:
# about an hour...
def extract_metadata(file_path, features):
    
    global series_uid, ext_feat
    
    ds = pydicom.dcmread(file_path, stop_before_pixels=True)
        
    file_metadata  = {
        'path': file_path,
    }

    cur_series_uid = file_path.split('/')[-2]
    
    if series_uid == cur_series_uid:
        file_metadata |= ext_feat
    else:
        series_uid = cur_series_uid
        ext_feat = ext_data[cur_series_uid]
        
        file_metadata |= ext_feat
    
    for feat in features:
        file_metadata[feat]  =  getattr(ds, feat, None)
    
    metadata_lst.append(file_metadata)    


metadata_lst = []
series_uid = ''
ext_feat = {}

_ = thread_map(
    lambda path: extract_metadata(path, feat_list), [dcm_file for dcm_file in file_list],
    tqdm_class = async_tqdm, 
    max_workers = 32,
    total = len(file_list),
    ncols = 100
)

In [ ]:
metadata_df = pd.DataFrame(metadata_lst)

You can see the distribution of categorical features

In [ ]:

def feat_values_rate(name: str, prc_sign=True):
    if prc_sign:
        return (metadata_df[name].value_counts(normalize=True) * 100).round(1).astype(str) + '%'
    else:
        return (metadata_df[name].value_counts(normalize=True) * 100).round(3)

In [ ]:
feat_values_rate('Manufacturer')

Siemens, Philips, GE - undisputed leaders

In [ ]:
feat_values_rate('SeriesDescription')[:10]

13 percent is a mockery! And there's no data for another 5 percent. Otherwise, you can check it out occasionally if you have any doubts about the weightening, etc. Parsing this unbridled creativity is best done only as a last resort )

In [ ]:
feat_values_rate('ScanningSequence')

3.4% is "Unknown", that's about 28k images

Since I plan to use the `Anatomical_Plane`, `Fat_Suppression`, `ScanningSequence`, `RepetitionTime`, and `EchoTime` features during preprocessing to distribute images across subnets, let's take a closer look at their contents

In [ ]:
metadata_df.loc[:, 'ScanningSequence'] = metadata_df['ScanningSequence'].astype(str)
res = (metadata_df
       .groupby(['Anatomical_Plane', 'Fat_Suppression', 'ScanningSequence'])
       .agg({'RepetitionTime': ['min', 'max'], 'EchoTime': ['min', 'max', 'size']})
      )
res.style.format(
    {
        ('RepetitionTime', 'min'): '{:,.2f}'.format,
        ('RepetitionTime', 'max'): '{:,.2f}'.format,
        ('EchoTime', 'min'): '{:,.2f}'.format,
        ('EchoTime', 'max'): '{:,.2f}'.format,
        ('EchoTime', 'size'): '{:,d}'.format,
    }
)

Well, we'll analyze this in more detail during preprocessing, but for now, let's note the missing data for approximately 28,000 images. We'll also consider how this can be filled by parsing SeriesDescription.

In [ ]:
metadata_df[metadata_df['ScanningSequence']=='None'][['Fat_Suppression', 'SeriesDescription']].value_counts()

That's not bad! At least the missing `ScanningSequence`, `RepetitionTime`, and `EchoTime` don't overlap with the missing `SeriesDescription`

In [ ]:
# we need to convert the MultiValue type to a list, before saving
mv_cols = ['PixelSpacing', 'ImagePositionPatient']

for col in mv_cols:
    metadata_df[col] = metadata_df[col].apply(
        lambda x: list(x) if x is not None and hasattr(x, '__iter__') else x
    )

# let's save it for the future
metadata_df.to_parquet('metadata.parquet', index=False)

We've gotten a bit deep, time to move on to pre-processing. Stay tuned!